In [1]:
import os

# Ensure protobuf uses the pure‑python implementation compatible with TF 2.18
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt



2026-05-07 02:21:56.490021: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778120516.504935      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778120516.509499      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-07 02:21:56.529922: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
BASE_PATH = "/kaggle/input/cassava-leaf-disease-classification"
TRAIN_IMG_PATH = os.path.join(BASE_PATH, "train_images")
TEST_IMG_PATH = os.path.join(BASE_PATH, "test_images")
TRAIN_CSV = os.path.join(BASE_PATH, "train.csv")
SAMPLE_SUB = os.path.join(BASE_PATH, "sample_submission.csv")

train_df = pd.read_csv(TRAIN_CSV)

# Convert labels to strings for Keras categorical mode
train_df["label"] = train_df["label"].astype(str)

train_df["filepath"] = train_df["image_id"].apply(
    lambda x: os.path.join(TRAIN_IMG_PATH, x)
)

datagen = ImageDataGenerator(rescale=1.0 / 255, validation_split=0.1)

train_gen = datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col="filepath",
    y_col="label",
    target_size=(128, 128),
    batch_size=32,
    class_mode="categorical",
    subset="training",
    shuffle=True,
    seed=42,
)

val_gen = datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col="filepath",
    y_col="label",
    target_size=(128, 128),
    batch_size=32,
    class_mode="categorical",
    subset="validation",
    shuffle=False,
    seed=42,
)


def build_model(num_classes=5):
    inputs = keras.Input(shape=(128, 128, 3))
    x = layers.Conv2D(32, 3, activation="relu")(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, activation="relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
    )
    return model


model = build_model(num_classes=5)

model.fit(train_gen, validation_data=val_gen, epochs=3, verbose=2)



Found 16849 validated image filenames belonging to 5 classes.
Found 1872 validated image filenames belonging to 5 classes.


2026-05-07 02:22:06.338080: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1778120526.338558      55 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:05:00.0, compute capability: 8.6


Epoch 1/3


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
I0000 00:00:1778120528.457552     108 service.cc:148] XLA service 0x7f5f38009fe0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778120528.457605     108 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-05-07 02:22:08.479793: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778120528.581770     108 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1778120530.250919     108 device_compiler.h:188] Compile

527/527 - 50s - 94ms/step - accuracy: 0.6183 - loss: 1.1357 - val_accuracy: 0.6303 - val_loss: 1.0408
Epoch 2/3
527/527 - 42s - 80ms/step - accuracy: 0.6305 - loss: 1.0467 - val_accuracy: 0.6458 - val_loss: 1.0360
Epoch 3/3
527/527 - 42s - 80ms/step - accuracy: 0.6387 - loss: 1.0113 - val_accuracy: 0.6533 - val_loss: 0.9430


In [3]:
sample_sub = pd.read_csv(SAMPLE_SUB)

sample_sub["filepath"] = sample_sub["image_id"].apply(
    lambda x: os.path.join(TEST_IMG_PATH, x)
)

preds = []
for idx, row in sample_sub.iterrows():
    img_path = row["filepath"]
    img = image.load_img(img_path, target_size=(128, 128))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)  # (1, 128, 128, 3)
    prediction = model.predict(img_array, verbose=0)
    pred_label = int(np.argmax(prediction, axis=1)[0])
    preds.append(pred_label)

submission = pd.DataFrame({"image_id": sample_sub["image_id"], "label": preds})
submission.to_csv("submission.csv", index=False)
print("Submission saved to submission.csv")

Submission saved to submission.csv
